In [1]:
import tarfile
import warnings
from glob import glob
from tqdm import tqdm

import anndata as ad
import matplotlib.pyplot as plt
import muon as mu
import pandas as pd
import numpy as np
import scanpy as sc
import scirpy as ir

sc.set_figure_params(figsize=(4, 4))
sc.settings.verbosity = 2 # verbosity: errors (0), warnings (1), info (2), hints (3)
pd.set_option('display.max_rows', 100)

# Read TCR 

In [2]:
files = glob('/home/liyanguo/MyImmuCell/01_rawdata/MyImmuCell_TCR_out/*')
len(files)

1998

In [3]:
adata_tcr = {}
for i in tqdm(files):
    sample_name = i.split('/')[-1].split('.')[0].split('_TCR')[0]
    airr_data = pd.read_csv(i)
    airr_data.columns=['cell_id', 'locus', 'v_call', 'd_call', 'j_call', 'c_call',
       'full_length', 'productive', 'junction_aa', 'junction', 'reads', 'umi_count','raw_clonotype_id']
    airr_data['cell_id'] = [sample_name + "_" + name for name in airr_data['cell_id']]
    adata = ir.io.read_airr(airr_data)
    adata_tcr[sample_name] = adata

  0%|                                                                                                     | 0/1998 [00:00<?, ?it/s]/home/liyanguo/anaconda3/envs/R/lib/python3.12/site-packages/anndata/utils.py:349: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)
100%|██████████████████████████████████████████████████████████████████████████████████████████| 1998/1998 [12:23<00:00,  2.69it/s]


In [4]:
del adata,airr_data

In [5]:
adata = ad.concat(adata_tcr)

In [6]:
adata.write_h5ad("/home/liyanguo/MyImmuCell/02_Read_QC/scTCR_MyImmuCell.h5ad",compression="gzip")

## TCR chain_qc

In [7]:
ir.pp.index_chains(adata)
ir.tl.chain_qc(adata)

Filtering chains...
Indexing VJ chains...
Indexing VDJ chains...
build result array
Stored result in `adata.obs["receptor_type"]`.
Stored result in `adata.obs["receptor_subtype"]`.
Stored result in `adata.obs["chain_pairing"]`.


In [8]:
data_tcr = ir.get.airr(adata, ["locus","v_call","j_call","junction_aa","junction","umi_count"],chain=('VJ_1', 'VDJ_1'))

In [9]:
data_tcr = adata.obs.join(data_tcr, how='left')

In [10]:
data_tcr['receptor_type'].value_counts()

receptor_type
TCR    16153570
Name: count, dtype: int64

In [11]:
data_tcr.to_csv("/home/liyanguo/MyImmuCell/02_Read_QC/scTCR_MyImmuCell.csv")

In [12]:
data_tcr

,receptor_type,receptor_subtype,chain_pairing,VJ_1_locus,VJ_1_v_call,VJ_1_j_call,VJ_1_junction_aa,VJ_1_junction,VJ_1_umi_count,VDJ_1_locus,VDJ_1_v_call,VDJ_1_j_call,VDJ_1_junction_aa,VDJ_1_junction,VDJ_1_umi_count
cell_id,,,,,,,,,,,,,,,
D0057_Rep2_AAACATCG_AAGGACAC_AACGCTTA,TCR,TRA+TRB,single pair,TRA,TRAV19,TRAJ17,CAHKAAGNKLTF,TGTGCCCACAAAGCTGCAGGCAACAAGCTAACTTTT,11,TRB,TRBV27,TRBJ1-4,CASSLYFPGTSTTNEKLFF,TGTGCCAGCAGTTTGTACTTCCCCGGGACATCCACAACTAATGAAA...,48
D0057_Rep2_AAACATCG_AGCCATGC_ACCTCCAA,TCR,TRA+TRB,single pair,TRA,TRAV19,TRAJ17,CAHKAAGNKLTF,TGTGCCCACAAAGCTGCAGGCAACAAGCTAACTTTT,4,TRB,TRBV27,TRBJ1-4,CASSLYFPGTSTTNEKLFF,TGTGCCAGCAGTTTGTACTTCCCCGGGACATCCACAACTAATGAAA...,17
D0057_Rep2_AAACATCG_CTAAGGTC_AGCACCTC,TCR,TRA+TRB,single pair,TRA,TRAV19,TRAJ17,CAHKAAGNKLTF,TGTGCCCACAAAGCTGCAGGCAACAAGCTAACTTTT,10,TRB,TRBV27,TRBJ1-4,CASSLYFPGTSTTNEKLFF,TGTGCCAGCAGTTTGTACTTCCCCGGGACATCCACAACTAATGAAA...,1
D0057_Rep2_AAACATCG_TGAAGAGA_CAGCGTTA,TCR,TRA+TRB,single pair,TRA,TRAV19,TRAJ17,CAHKAAGNKLTF,TGTGCCCACAAAGCTGCAGGCAACAAGCTAACTTTT,1,TRB,TRBV27,TRBJ1-4,CASSLYFPGTSTTNEKLFF,TGTGCCAGCAGTTTGTACTTCCCCGGGACATCCACAACTAATGAAA...,32
D0057_Rep2_AACAACCA_AAGAGATC_CGCATACA,TCR,TRA+TRB,single pair,TRA,TRAV19,TRAJ17,CAHKAAGNKLTF,TGTGCCCACAAAGCTGCAGGCAACAAGCTAACTTTT,4,TRB,TRBV27,TRBJ1-4,CASSLYFPGTSTTNEKLFF,TGTGCCAGCAGTTTGTACTTCCCCGGGACATCCACAACTAATGAAA...,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
D0037_Rep1_CCTAATCC_TCCGTCTA_ACAAGCTA,TCR,TRA+TRB,orphan VDJ,None,None,None,None,None,None,TRB,TRBV10-3,TRBJ1-2,CAIKGTESLNYGYTF,TGTGCCATCAAGGGGACAGAGAGTCTTAACTATGGCTACACCTTC,5
D0037_Rep1_ACAGATTC_AGATGTAC_ACAGATTC,TCR,TRA+TRB,orphan VDJ,None,None,None,None,None,None,TRB,TRBV10-3,TRBJ1-3,CAIGTGAGNTIYF,TGTGCCATTGGAACAGGGGCTGGAAACACCATATATTTT,5
D0037_Rep1_AAGAGATC_TTCACGCA_GTACGCAA,TCR,TRA+TRB,orphan VDJ,None,None,None,None,None,None,TRB,TRBV15,TRBJ2-1,CAGLAGERDEQFF,TGTGCCGGGCTAGCGGGAGAAAGGGATGAGCAGTTCTTC,17


In [13]:
del data_tcr,adata_tcr

# Read BCR

In [14]:
files = glob('/home/liyanguo/MyImmuCell/01_rawdata/MyImmuCell_BCR_out/*')
len(files)

1998

In [15]:
adata_bcr = {}
for i in tqdm(files):
    sample_name = i.split('/')[-1].split('.')[0].split('_BCR')[0]
    airr_data = pd.read_csv(i)
    airr_data.columns=['cell_id', 'locus', 'v_call', 'd_call', 'j_call', 'c_call',
       'full_length', 'productive', 'junction_aa', 'junction', 'reads', 'umi_count','raw_clonotype_id']
    airr_data['cell_id'] = [sample_name + "_" + name for name in airr_data['cell_id']]
    adata = ir.io.read_airr(airr_data)
    adata_bcr[sample_name] = adata

100%|██████████████████████████████████████████████████████████████████████████████████████████| 1998/1998 [02:57<00:00, 11.25it/s]


In [16]:
del adata,airr_data

In [17]:
adata = ad.concat(adata_bcr)

In [18]:
adata.write_h5ad("/home/liyanguo/MyImmuCell/02_Read_QC/scBCR_MyImmuCell.h5ad",compression="gzip")

## BCR chain_qc

In [19]:
ir.pp.index_chains(adata)
ir.tl.chain_qc(adata)

Filtering chains...
Indexing VJ chains...
Indexing VDJ chains...
build result array
Stored result in `adata.obs["receptor_type"]`.
Stored result in `adata.obs["receptor_subtype"]`.
Stored result in `adata.obs["chain_pairing"]`.


In [20]:
data_bcr = ir.get.airr(adata, ["locus","v_call","j_call","junction_aa","junction","umi_count"],chain=('VJ_1', 'VDJ_1'))

In [21]:
data_bcr = adata.obs.join(data_bcr, how='left')

In [22]:
data_bcr['receptor_type'].value_counts()

receptor_type
BCR    3662048
Name: count, dtype: int64

In [23]:
data_bcr.to_csv("/home/liyanguo/MyImmuCell/02_Read_QC/scBCR_MyImmuCell.csv")

In [24]:
data_bcr

,receptor_type,receptor_subtype,chain_pairing,VJ_1_locus,VJ_1_v_call,VJ_1_j_call,VJ_1_junction_aa,VJ_1_junction,VJ_1_umi_count,VDJ_1_locus,VDJ_1_v_call,VDJ_1_j_call,VDJ_1_junction_aa,VDJ_1_junction,VDJ_1_umi_count
cell_id,,,,,,,,,,,,,,,
D0502_Rep1_AACGCTTA_ACAAGCTA_ATTGAGGA,BCR,IGH,orphan VDJ,None,None,None,None,None,None,IGH,IGHV3-48,IGHJ4,CSRDADGDHDFDYW,TGTTCGAGAGATGCTGACGGTGACCACGACTTTGACTACTGG,7
D0502_Rep1_ACCACTGT_AACAACCA_AAGAGATC,BCR,IGH,orphan VDJ,None,None,None,None,None,None,IGH,IGHV3-48,IGHJ4,CSRDADGDHDFDYW,TGTTCGAGAGATGCTGACGGTGACCACGACTTTGACTACTGG,7
D0502_Rep1_ACCTCCAA_GCTAACGA_CGAACTTA,BCR,IGH,orphan VDJ,None,None,None,None,None,None,IGH,IGHV3-48,IGHJ4,CSRDADGDHDFDYW,TGTTCGAGAGATGCTGACGGTGACCACGACTTTGACTACTGG,199
D0502_Rep1_ACGTATCA_ATCATTCC_AACAACCA,BCR,IGH,orphan VDJ,None,None,None,None,None,None,IGH,IGHV3-48,IGHJ4,CSRDADGDHDFDYW,TGTTCGAGAGATGCTGACGGTGACCACGACTTTGACTACTGG,24
D0502_Rep1_ACTATGCA_CAGCGTTA_CTAAGGTC,BCR,IGH,orphan VDJ,None,None,None,None,None,None,IGH,IGHV3-48,IGHJ4,CSRDADGDHDFDYW,TGTTCGAGAGATGCTGACGGTGACCACGACTTTGACTACTGG,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
D0942_Rep2_ACGTATCA_ACACAGAA_AGAGTCAA,BCR,IGH+IGK,orphan VJ,IGK,IGKV4-1,IGKJ2,CHQYYSLPYTF,TGTCACCAATATTACAGTCTTCCGTACACTTTT,6,None,None,None,None,None,None
D0942_Rep2_GGAGAACA_CGACTGGA_TGGAACAA,BCR,IGH+IGK,orphan VJ,IGK,IGKV4-1,IGKJ1,CHQYWNLWTF,TGTCACCAATATTGGAATCTGTGGACGTTC,15,None,None,None,None,None,None
D0942_Rep2_AGAGTCAA_CTGTAGCC_ATAGCGAC,BCR,IGH+IGK,orphan VJ,IGK,IGKV6-21,IGKJ2,CHQTITSTFTF,TGTCATCAAACCATCACTTCAACGTTCACTTTC,135,None,None,None,None,None,None


In [25]:
del data_bcr,adata_bcr